# Prophet Forecasting Playground

This notebook provides an interactive environment to test the cluster-based Prophet model adapted for Retail Demand Forecasting.

### Key Features
- **Retail Specific**: Uses `np.log1p` transformation and Ridge-like Prophet configurations to handle extreme demand skewness.
- **Cluster-based**: Fits independent Prophet models for each `profile_cluster_id`.
- **Symmetric Evaluation**: Evaluates performance using SMAPE and MAE, which are robust to zero-demand weeks.
- **Modular Architecture**: Uses decoupled functions from `src/models/prophet_model.py`.


## 0. Environment Setup

Resolve project-level modules from the `src` directory and load the necessary forecasting functions.

In [23]:
%load_ext autoreload
%autoreload 2

import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root is in path
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Import specialized Prophet functions
from src.models.prophet_model import (
    load_processed_data, 
    preprocess_and_split, 
    train_models, 
    predict_models, 
    evaluate_models,
    save_prophet_artifacts
)
from src.tools.visualization import plot_cluster_portfolio, analyze_time_periods

print("Setup complete. Local prophet modules loaded from src/.")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Setup complete. Local prophet modules loaded from src/.


## 1. Global Data Loading

We load the global processed dataset once. It contains all features, including the raw consumption and weather signals that will be scaled differently depending on the forecast mode.

In [24]:
dataset_path = os.path.join(PROJECT_ROOT, 'data', 'processed_retail_data.parquet')
df_long = load_processed_data(dataset_path)

print(f"\n Total observations loaded: {len(df_long):,}")

Loading processed data...

 Total observations loaded: 41,548,234


## 2. Train & Evaluate Prophet Models

We split the data chronologically, aggregate the training data by cluster to train the Prophet models, and then evaluate the predictions at the cluster level.

In [ ]:
# A.1 Prepare Data
train_agg, test_agg, test_raw = preprocess_and_split(df_long)

# A.2 Train Cluster Models
cluster_models = train_models(train_agg)

# A.3 Predict & Un-scale
test_raw = predict_models(cluster_models, test_agg, test_raw)

# A.4 Evaluate
portfolio_eval, summary = evaluate_models(test_raw)
display(summary)

## 3. Visualize Performance

We plot the overall global aggregate and the individual cluster profiles.

In [ ]:
plot_cluster_portfolio(portfolio_eval, summary, model_label="Prophet Model")
analyze_time_periods(test_raw)